# 2.2 - Applicazione del classificatore Naive-Bayes al file `manuale.csv`
Applichiamo il costruttore di Naive-Bayes al file `manuale.csv` (14 campioni bilanciati) prima manualmente e poi implementando il classificatore in python

## 1. Cenni di Teoria: Regola di Bayes della probabilità condizionata
La classificazione con Naive Bayes si basa sulla **regola di Bayes** della probabilità condizionata. La regola di Bayes dice che se si dispone di un'ipotesi H e di prove E che riguardano tale ipotesi, allora:

$$P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}$$

- $P(H)$ è la **probabilità a priori** dell'ipotesi H;
- $P(E \mid H)$ è la probabilità di E condizionata dall'ipotesi H;
- $P(E)$ è uguale per tutte le classi, quindi **scompare nella normalizzazione** finale.

L'evidenza $E$ è composta da 4 "prove" ($E_0 \dots E_3$, una per feature). L'assunzione **naive** è che siano **indipendenti tra loro: questo permette di scomporre $P(E \mid H)$ in un prodotto di probabilità semplici, stimabili contando le frequenze:

$$P(E \mid H) = P(E_0 \mid H) \cdot P(E_1 \mid H) \cdot P(E_2 \mid H) \cdot P(E_3 \mid H)$$

Quindi calcoliamo uno *score*:

$$score(c) = P(c) \cdot \prod_{i=0}^{3} P(f_i = v_i \mid c)$$

e le probabilità finali si ottengono **normalizzando** (la somma deve fare 1):

$$P(1 \mid E) = \frac{score(1)}{score(1) + score(0)} \qquad P(0 \mid E) = \frac{score(0)}{score(1) + score(0)}$$

La classe predetta (LABEL=1 o LABEL=0) è quella con lo score maggiore. L'ipotesi di indipendenza è certamente semplicistica nella vita reale, ma Naive Bayes funziona in modo molto efficace su dati reali, spesso superando classificatori più sofisticati.

**Perché l'approccio a conteggi (e non gaussiano)?** Le 4 feature, pur essendo numeriche, hanno una cardinalità molto bassa sull'intero dataset (26, 16, 87 e 164 valori distinti su ~412.000 righe, verificato nel notebook 01): si comportano di fatto come attributi categorici.

## 2. Miglioria: Lo stimatore di Laplace
**Il problema della frequenza zero.** Le probabilità $P(f_i = v \mid c)$ sono stimate contando le occorrenze nel file `manuale.csv`. Se un valore $v$ non compare **mai** insieme a una classe $c$, la stima vale $\frac{0}{n} = 0$; e poiché lo score è un **prodotto**, un singolo fattore nullo azzera l'intero score, cancellando il contributo di tutte le altre feature.

**Lo stimatore di Laplace.** La correzione standard consiste nell'aggiungere 1 a ogni conteggio al numeratore e, per compensare, il numero di istanze della feature al denominatore:

$$P(f_i = v \mid c) = \frac{count(v, c) + 1}{n_c + k_i}$$

dove $n_c$ è il numero di campioni della classe $c$ e $k_i$ il numero di valori distinti della feature $i$-esima. Così ogni valore, anche mai osservato, riceve una probabilità piccola ma **non nulla**, e la somma su tutti i valori possibili resta 1.

**Scelta di $k$.** Usiamo la cardinalità calcolata sull'**intero dataset** (notebook 01) e non sui soli 14 campioni: $k_0 = 26$, $k_1 = 16$, $k_2 = 87$, $k_3 = 164$. Stimare $k$ sul solo `manuale.csv` sottostimerebbe lo spazio dei valori possibili, non riuscendo così a coprire valori ancora non visti.

## 3. Implementazione manuale del classificatore Naive-Bayes per il file `manuale.csv`
Stimiamo le probabilità contando le frequenze sui 14 campioni di `manuale.csv` e valutiamo il classificatore sullo stesso file. 
Tutte le frazioni includono già la correzione di Laplace.

### 3.1 Tabelle delle frequenze (stile Tabella 4.2 delle slide)

Il file `manuale.csv` contiene **14 campioni bilanciati**: 7 con `LABEL=1` (dropout) e 7 con `LABEL=0`.

Per ogni feature contiamo quante volte ciascun valore compare in ognuna delle due classi, e accanto riportiamo la probabilità condizionata **già corretta con lo stimatore di Laplace**.

**FEATURE0** &nbsp; ($k = 26$)

| Valore | conteggio (LABEL=1) | conteggio (LABEL=0) | $P(v\mid 1)$ | $P(v\mid 0)$ |
|---|---|---|---|---|
| -0.3200 | 5 | 5 | $\frac{5+1}{7+26}=\frac{6}{33}$ | $\frac{5+1}{7+26}=\frac{6}{33}$ |
| 3.7243 | 1 | 1 | $\frac{1+1}{7+26}=\frac{2}{33}$ | $\frac{1+1}{7+26}=\frac{2}{33}$ |
| 5.0723 | 1 | 0 | $\frac{1+1}{7+26}=\frac{2}{33}$ | $\frac{0+1}{7+26}=\frac{1}{33}$ |
| 10.4647 | 0 | 1 | $\frac{0+1}{7+26}=\frac{1}{33}$ | $\frac{1+1}{7+26}=\frac{2}{33}$ |

**FEATURE1** &nbsp; ($k = 16$)

| Valore | conteggio (LABEL=1) | conteggio (LABEL=0) | $P(v\mid 1)$ | $P(v\mid 0)$ |
|---|---|---|---|---|
| -0.4357 | 6 | 7 | $\frac{6+1}{7+16}=\frac{7}{23}$ | $\frac{7+1}{7+16}=\frac{8}{23}$ |
| 2.1087 | 1 | 0 | $\frac{1+1}{7+16}=\frac{2}{23}$ | $\frac{0+1}{7+16}=\frac{1}{23}$ |

**FEATURE2** &nbsp; ($k = 87$)

| Valore | conteggio (LABEL=1) | conteggio (LABEL=0) | $P(v\mid 1)$ | $P(v\mid 0)$ |
|---|---|---|---|---|
| -0.3942 | 3 | 2 | $\frac{3+1}{7+87}=\frac{4}{94}$ | $\frac{2+1}{7+87}=\frac{3}{94}$ |
| 0.6078 | 2 | 1 | $\frac{2+1}{7+87}=\frac{3}{94}$ | $\frac{1+1}{7+87}=\frac{2}{94}$ |
| 1.6098 | 1 | 0 | $\frac{1+1}{7+87}=\frac{2}{94}$ | $\frac{0+1}{7+87}=\frac{1}{94}$ |
| 2.1109 | 1 | 0 | $\frac{1+1}{7+87}=\frac{2}{94}$ | $\frac{0+1}{7+87}=\frac{1}{94}$ |
| 4.1150 | 0 | 1 | $\frac{0+1}{7+87}=\frac{1}{94}$ | $\frac{1+1}{7+87}=\frac{2}{94}$ |
| 5.1170 | 0 | 1 | $\frac{0+1}{7+87}=\frac{1}{94}$ | $\frac{1+1}{7+87}=\frac{2}{94}$ |
| 10.6282 | 0 | 1 | $\frac{0+1}{7+87}=\frac{1}{94}$ | $\frac{1+1}{7+87}=\frac{2}{94}$ |
| 43.1946 | 0 | 1 | $\frac{0+1}{7+87}=\frac{1}{94}$ | $\frac{1+1}{7+87}=\frac{2}{94}$ |

**FEATURE3** &nbsp; ($k = 164$)

| Valore | conteggio (LABEL=1) | conteggio (LABEL=0) | $P(v\mid 1)$ | $P(v\mid 0)$ |
|---|---|---|---|---|
| -0.0673 | 1 | 0 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{0+1}{7+164}=\frac{1}{171}$ |
| 0.1334 | 0 | 1 | $\frac{0+1}{7+164}=\frac{1}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |
| 0.5348 | 1 | 1 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |
| 0.7355 | 1 | 0 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{0+1}{7+164}=\frac{1}{171}$ |
| 0.9362 | 1 | 0 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{0+1}{7+164}=\frac{1}{171}$ |
| 1.1369 | 1 | 1 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |
| 1.3376 | 0 | 1 | $\frac{0+1}{7+164}=\frac{1}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |
| 1.7390 | 1 | 1 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |
| 2.1403 | 1 | 1 | $\frac{1+1}{7+164}=\frac{2}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |
| 7.7598 | 0 | 1 | $\frac{0+1}{7+164}=\frac{1}{171}$ | $\frac{1+1}{7+164}=\frac{2}{171}$ |

**Probabilità a priori**: $P(1)=\frac{7}{14}$, $P(0)=\frac{7}{14}$.


### 3.2 Calcolo delle probabilità per ciascuna azione

Per ogni azione moltiplichiamo le probabilità condizionate delle 4 feature (lette dalle tabelle sopra) per la probabilità a priori della classe, ottenendo i due *score*; poi normalizziamo dividendo per la loro somma.

---

**Azione 1 — ACTIONID 23138**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 5.1170 | 2.1403 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{1}{94} \cdot \tfrac{2}{171} \approx 3.4426 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 7.8687 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{3.4426 \cdot 10^{-6}}{3.4426 \cdot 10^{-6} + 7.8687 \cdot 10^{-6}} = 30.43\% \qquad P(0\mid E) = 69.57\%$$

Predizione: **0** — classificazione corretta.

---

**Azione 2 — ACTIONID 229525**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 10.6282 | 0.1334 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{1}{94} \cdot \tfrac{1}{171} \approx 1.7213 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 7.8687 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{1.7213 \cdot 10^{-6}}{1.7213 \cdot 10^{-6} + 7.8687 \cdot 10^{-6}} = 17.95\% \qquad P(0\mid E) = 82.05\%$$

Predizione: **0** — classificazione corretta.

---

**Azione 3 — ACTIONID 3508**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 0.6078 | 0.7355 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{3}{94} \cdot \tfrac{2}{171} \approx 1.0328 \cdot 10^{-5}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{1}{171} \approx 3.9344 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{1.0328 \cdot 10^{-5}}{1.0328 \cdot 10^{-5} + 3.9344 \cdot 10^{-6}} = 72.41\% \qquad P(0\mid E) = 27.59\%$$

Predizione: **1** — classificazione corretta.

---

**Azione 4 — ACTIONID 156007**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 43.1946 | 0.5348 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{1}{94} \cdot \tfrac{2}{171} \approx 3.4426 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 7.8687 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{3.4426 \cdot 10^{-6}}{3.4426 \cdot 10^{-6} + 7.8687 \cdot 10^{-6}} = 30.43\% \qquad P(0\mid E) = 69.57\%$$

Predizione: **0** — classificazione corretta.

---

**Azione 5 — ACTIONID 391690**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 2.1109 | 0.9362 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 6.8852 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{1}{94} \cdot \tfrac{1}{171} \approx 1.9672 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{6.8852 \cdot 10^{-6}}{6.8852 \cdot 10^{-6} + 1.9672 \cdot 10^{-6}} = 77.78\% \qquad P(0\mid E) = 22.22\%$$

Predizione: **1** — classificazione corretta.

---

**Azione 6 — ACTIONID 102135**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | 10.4647 | -0.4357 | -0.3942 | 1.7390 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{1}{33} \cdot \tfrac{7}{23} \cdot \tfrac{4}{94} \cdot \tfrac{2}{171} \approx 2.2951 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{2}{33} \cdot \tfrac{8}{23} \cdot \tfrac{3}{94} \cdot \tfrac{2}{171} \approx 3.9344 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{2.2951 \cdot 10^{-6}}{2.2951 \cdot 10^{-6} + 3.9344 \cdot 10^{-6}} = 36.84\% \qquad P(0\mid E) = 63.16\%$$

Predizione: **0** — classificazione corretta.

---

**Azione 7 — ACTIONID 58281**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | 3.7243 | -0.4357 | -0.3942 | 1.1369 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{2}{33} \cdot \tfrac{7}{23} \cdot \tfrac{4}{94} \cdot \tfrac{2}{171} \approx 4.5901 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{2}{33} \cdot \tfrac{8}{23} \cdot \tfrac{3}{94} \cdot \tfrac{2}{171} \approx 3.9344 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{4.5901 \cdot 10^{-6}}{4.5901 \cdot 10^{-6} + 3.9344 \cdot 10^{-6}} = 53.85\% \qquad P(0\mid E) = 46.15\%$$

Predizione: **1** — classificazione corretta.

---

**Azione 8 — ACTIONID 304813**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 1.6098 | 1.7390 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 6.8852 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{1}{94} \cdot \tfrac{2}{171} \approx 3.9344 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{6.8852 \cdot 10^{-6}}{6.8852 \cdot 10^{-6} + 3.9344 \cdot 10^{-6}} = 63.64\% \qquad P(0\mid E) = 36.36\%$$

Predizione: **1** — classificazione corretta.

---

**Azione 9 — ACTIONID 53632**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 4.1150 | 1.3376 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{1}{94} \cdot \tfrac{1}{171} \approx 1.7213 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 7.8687 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{1.7213 \cdot 10^{-6}}{1.7213 \cdot 10^{-6} + 7.8687 \cdot 10^{-6}} = 17.95\% \qquad P(0\mid E) = 82.05\%$$

Predizione: **0** — classificazione corretta.

---

**Azione 10 — ACTIONID 78129**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | 5.0723 | -0.4357 | -0.3942 | 2.1403 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{2}{33} \cdot \tfrac{7}{23} \cdot \tfrac{4}{94} \cdot \tfrac{2}{171} \approx 4.5901 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{1}{33} \cdot \tfrac{8}{23} \cdot \tfrac{3}{94} \cdot \tfrac{2}{171} \approx 1.9672 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{4.5901 \cdot 10^{-6}}{4.5901 \cdot 10^{-6} + 1.9672 \cdot 10^{-6}} = 70.00\% \qquad P(0\mid E) = 30.00\%$$

Predizione: **1** — classificazione corretta.

---

**Azione 11 — ACTIONID 1226**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | 3.7243 | -0.4357 | -0.3942 | 1.1369 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{2}{33} \cdot \tfrac{7}{23} \cdot \tfrac{4}{94} \cdot \tfrac{2}{171} \approx 4.5901 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{2}{33} \cdot \tfrac{8}{23} \cdot \tfrac{3}{94} \cdot \tfrac{2}{171} \approx 3.9344 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{4.5901 \cdot 10^{-6}}{4.5901 \cdot 10^{-6} + 3.9344 \cdot 10^{-6}} = 53.85\% \qquad P(0\mid E) = 46.15\%$$

Predizione: **1** — classificazione **errata**.

---

**Azione 12 — ACTIONID 34413**  (LABEL reale: **0**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 0.6078 | 7.7598 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{3}{94} \cdot \tfrac{1}{171} \approx 5.1639 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 7.8687 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{5.1639 \cdot 10^{-6}}{5.1639 \cdot 10^{-6} + 7.8687 \cdot 10^{-6}} = 39.62\% \qquad P(0\mid E) = 60.38\%$$

Predizione: **0** — classificazione corretta.

---

**Azione 13 — ACTIONID 135**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | 2.1087 | -0.3942 | -0.0673 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{2}{23} \cdot \tfrac{4}{94} \cdot \tfrac{2}{171} \approx 3.9344 \cdot 10^{-6}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{1}{23} \cdot \tfrac{3}{94} \cdot \tfrac{1}{171} \approx 7.3769 \cdot 10^{-7}$$

$$P(1\mid E) = \frac{3.9344 \cdot 10^{-6}}{3.9344 \cdot 10^{-6} + 7.3769 \cdot 10^{-7}} = 84.21\% \qquad P(0\mid E) = 15.79\%$$

Predizione: **1** — classificazione corretta.

---

**Azione 14 — ACTIONID 27044**  (LABEL reale: **1**)

| | FEAT0 | FEAT1 | FEAT2 | FEAT3 |
|---|---|---|---|---|
| valore | -0.3200 | -0.4357 | 0.6078 | 0.5348 |

$$score(1) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{7}{23} \cdot \tfrac{3}{94} \cdot \tfrac{2}{171} \approx 1.0328 \cdot 10^{-5}$$

$$score(0) = \tfrac{7}{14} \cdot \tfrac{6}{33} \cdot \tfrac{8}{23} \cdot \tfrac{2}{94} \cdot \tfrac{2}{171} \approx 7.8687 \cdot 10^{-6}$$

$$P(1\mid E) = \frac{1.0328 \cdot 10^{-5}}{1.0328 \cdot 10^{-5} + 7.8687 \cdot 10^{-6}} = 56.76\% \qquad P(0\mid E) = 43.24\%$$

Predizione: **1** — classificazione corretta.


### 3.3 Riepilogo delle prestazioni

| # | ACTIONID | LABEL reale | Predizione | Esito |
|---|---|---|---|---|
| 1 | 23138 | 0 | 0 | ✅ |
| 2 | 229525 | 0 | 0 | ✅ |
| 3 | 3508 | 1 | 1 | ✅ |
| 4 | 156007 | 0 | 0 | ✅ |
| 5 | 391690 | 1 | 1 | ✅ |
| 6 | 102135 | 0 | 0 | ✅ |
| 7 | 58281 | 1 | 1 | ✅ |
| 8 | 304813 | 1 | 1 | ✅ |
| 9 | 53632 | 0 | 0 | ✅ |
| 10 | 78129 | 1 | 1 | ✅ |
| 11 | 1226 | 0 | 1 | ❌ |
| 12 | 34413 | 0 | 0 | ✅ |
| 13 | 135 | 1 | 1 | ✅ |
| 14 | 27044 | 1 | 1 | ✅ |

**Accuratezza = 13/14 = 92.86%**

**Osservazione:** lo stimatore di Laplace è stato essenziale: molti valori di FEATURE2/FEATURE3 compaiono in una sola classe e senza correzione avrebbero azzerato interi prodotti.


## 4. Implementazione in Python del classificatore Naive-Bayes per il file `manuale.csv`
Utilizziamo python e la sua libreria per la manipolazione di dati Pandas per implementare il classificatore Naive-Bayes sul nostro file di 14 campioni `manuale.csv`.
Replichiamo quindi quanto già fatto prima manualmente (sezione 3) ma adesso con poche righe di codice Python.

### 4.1. Setup del codice
Nella cella di codice sottostante andiamo a fare un setup utile a preparare un ambiente robusto per i prossimi passaggi.

In [1]:
import pandas as pd #ricordiamo che pandas è la libreria principale per la manipolazione dei dati in Python. Diamo il nome pd a pandas per comodità.

df = pd.read_csv("../data/manuale.csv") #uso la funzione read_csv di pandas e la memorizzo in df (dataframe) che è la struttura dati principale di pandas (una tabella con righe e colonne).

FEATS = ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"] #FEATS è una lista ordinata di feature, sarà utile per iterare sulle feature.

K = {"FEATURE0": 26, "FEATURE1": 16, "FEATURE2": 87, "FEATURE3": 164} #K è un dizionario che associa a ciascuna feature la sua cardinalità.

n1 = (df["LABEL"] == 1).sum()   # campioni dropout, con LABEL=1
n0 = (df["LABEL"] == 0).sum()   # campioni non-dropout, con LABEL=0
#df["LABEL"] == 1 o df["LABEL"] == 0 sono True se veri. In python sommare i true (con .sum()) equivale a contarli.
N = len(df) #numero totale di campioni in manuale.csv

### 4.2. Definizione delle funzioni
Nel blocco di codice sottostante andiamo a definire tre importanti funzioni:

- **prob_condizionata(feature, valore, label)**: Gli passiamo in input una feature, il suo valore e la Label (1=dropout e 0=non-dropout) per ottenere la probabilità condizionata.
- **score(azione, label)**: Per ciascuna azione (e per ciascuna label 0 o 1) calcoliamo lo score, cioè quel numero ottenuto moltiplicando alla probabilità a priori della label le varie probabilità cpndizionate. score sfrutta prob_condizionata per il calcolo delle varie probabilità condizionate.
- **classifica(azione)**: Classifichiamo un'azione data in 1 (dropout) o 0 (non dropout). classifica sfrutta score per ottenere quel numero che diventa poi la probabilità di una certa azione per una certa label.


In [2]:
def prob_condizionata(feature, valore, label): #otteniamo la probabilità condizionata per ciascuna feature, dato il valore della feature e la label.
    """P(feature = valore | label) con stimatore di Laplace.""" #""" = docstring, serve a documentare la funzione.
    if label == 1: 
        n_label = n1 #n1 è il numero di campioni con LABEL=1, cioè dropout
    else:
        n_label = n0 #n0 è il numero di campioni con LABEL=0, cioè non-dropout
    conteggio = ((df[feature] == valore) & (df["LABEL"] == label)).sum() #conteggio è il numero di campioni con la feature=valore e la LABEL=label(0 o 1).
    return (conteggio + 1) / (n_label + K[feature])

def score(azione, label): #calcoliamo lo score per ciascuna azione e per ciascuna sua label (1 o 0)
    """Probabilità a Priori * prodotto delle probabilita condizionate delle 4 feature."""
    risultato = (n1 if label == 1 else n0) / N #settiamo inizialmente il risultato come la probabilità a priori della label. N è il numero totale di campioni, n1 è il numero di campioni con LABEL=1, n0 è il numero di campioni con LABEL=0.
    for f in FEATS: # iteriamo sulle 4 feature
        risultato *= prob_condizionata(f, azione[f], label) #moltiplichiamo il risultato per la probabilità condizionata della feature f, dato il valore della feature nell'azione e la label.
    return risultato

def classifica(azione): #classifichiamo un'azione data
    """Restituisce P(classe=1) e P(classe=0) normalizzate e la classe predetta."""
    s1 = score(azione, 1) #calcoliamo lo score per la label=1 (dropout)
    s0 = score(azione, 0) #calcoliamo lo score per la label=0 (non-dropout)
    p1 = s1 / (s1 + s0)
    p0 = s0 / (s1 + s0)
    return p1, p0,(1 if s1 > s0 else 0)

### 4.3. Loop su tutte le azioni in `manuale.csv` e formattazione in tabella
Nel blocco di codice sottostante analizziamo ciascuna azione in `manuale.csv` e la classifichiamo grazie alla funzione precedentemente definita classifica(azione).
Inoltre inseriamo i risultati ottenuti in una tabella così da visualizzare il tutto graficamente.

In [3]:
righe = []
for _, azione in df.iterrows(): #con .iterrows() iteriamo sulle righe del dataframe df. In questo modo otteniamo per ciascuna riga un indice (che non ci serve) e un dizionario azione.
    p1, p0, pred = classifica(azione)
    righe.append({
        "ACTIONID": int(azione["ACTIONID"]),
        "LABEL_reale": int(azione["LABEL"]),
        "P(1) %": round(p1 * 100, 2), #scriviamo la probabilità di dropout come percentuale con 2 decimali
        "P(0) %": round(p0 * 100, 2), #scriviamo la probabilità di non-dropout come percentuale con 2 decimali
        "predizione": pred,
        "corretta": pred == azione["LABEL"], #se la predizione è uguale alla label reale, allora corretta è True, altrimenti False
    })

risultati = pd.DataFrame(righe)
risultati

,ACTIONID,LABEL_reale,P(1) %,P(0) %,predizione,corretta
0,23138,0,30.43,69.57,0,True
1,229525,0,17.95,82.05,0,True
2,3508,1,72.41,27.59,1,True
3,156007,0,30.43,69.57,0,True
4,391690,1,77.78,22.22,1,True
5,102135,0,36.84,63.16,0,True
6,58281,1,53.85,46.15,1,True
7,304813,1,63.64,36.36,1,True
8,53632,0,17.95,82.05,0,True
9,78129,1,70.00,30.00,1,True


### 4.4. Calcolo dell'accuratezza
Grazie alle righe di codice sottostanti calcoliamo l'accuratezza dei risultati ottenuti implementando il classificatore Naive-Bayes con Python.

In [4]:
accuratezza = risultati["corretta"].mean()
print(f"Accuratezza: {risultati['corretta'].sum()}/{N} = {accuratezza * 100:.2f}%")

Accuratezza: 13/14 = 92.86%


**Osservazione**: Abbiamo ottenuto gli stessi risultati e la stessa accuratezza ottenuti prima manualmente.